# Training a Pokémon DDPM from scratch (conditional version)

This notebook **hand-writes** a DDPM from scratch and trains it on 96×96 Pokémon sprites.
Compared to the unconditional version, this one feeds the **Pokémon name** to the network as a condition, which lets us:

1. **Generate a specified Pokémon by name**;
2. **Fuse two Pokémon** — interpolate the embeddings of the two names at sampling time (done in `pokemon_gen.ipynb`).

**No dependency on `diffusers`**, only `torch / torchvision`. The two things about diffusion models stay the same:
- **Forward noising**: gradually turn a clean image into pure noise (parameter-free math);
- **Reverse denoising**: train the UNet to learn how to wipe the noise away, so it can generate images.

In [ ]:
import math
import os, glob, json
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Time embedding

Encode the integer time step `t` (0 ~ T-1) into a vector that tells the network "how much noise there is right now".
We use the **sinusoidal / cosine positional encoding** from the Transformer.

In [ ]:
def sinusoidal_embedding(t, dim):
    """t: (B,) integer tensor  ->  (B, dim) float embedding"""
    half = dim // 2
    freqs = torch.exp(
        -math.log(10000) * torch.arange(half, device=t.device).float() / half
    )
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

## 2. How is the name condition added in? 🧬 (the key of this version)

The idea is very simple, exactly like the time embedding:

- Each Pokémon is a **category**; we learn a vector for it with `nn.Embedding`;
- **Add this name vector to the time vector** to form a single condition vector `c`;
- Inject `c` into every residual block. The network then knows two things at once:
  *"how much noise there is now"* and *"which Pokémon to draw"*.

We also reserve one extra **"null" category** (index = `num_classes`), used for
**classifier-free guidance (CFG)**: during training we replace the name with null with some probability,
so the network learns both "conditional" and "unconditional" predictions. At sampling time we use

$$\varepsilon = \varepsilon_{\text{uncond}} + w\,(\varepsilon_{\text{cond}} - \varepsilon_{\text{uncond}})$$

to amplify the conditional influence ($w$ = guidance strength). This both makes conditional generation more faithful
and is the foundation that makes **fusion** work — just interpolate the two name embeddings.

## 3. Basic building block: conditional residual block (ResBlock)

Each ResBlock: two passes of `GroupNorm→SiLU→Conv`, with the **condition vector `c`** (time + name)
added to the feature map in the middle, and a residual connection on the outside.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, c_dim, groups=8):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.GroupNorm(groups, in_ch), nn.SiLU(),
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
        )
        self.cond_mlp = nn.Sequential(           # condition vector -> out_ch
            nn.SiLU(), nn.Linear(c_dim, out_ch),
        )
        self.block2 = nn.Sequential(
            nn.GroupNorm(groups, out_ch), nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, c):
        h = self.block1(x)
        h = h + self.cond_mlp(c)[:, :, None, None]   # inject the "time+name" condition
        h = self.block2(h)
        return h + self.skip(x)

## 4. UNet backbone (conditional + self-attention + larger capacity)

The skeleton is still a UNet (96→48→24→12→24→48→96, stitched with skip connections). Compared to the previous version, this one increases capacity:
- **`base` channels 64 → 128**: the high-resolution layer (96) doubles from 64 to 128 channels, directly boosting the ability to render "details like facial features";
- **Place `n_blocks` ResBlocks at each level** (default 2, deeper);
- Use **`Stage`** to pack "several ResBlocks + optional self-attention" into one resolution level, making forward cleaner;
- Self-attention is still only placed at the low-resolution layers (24×24, 12×12);
- The `label_emb` + `cond()` name conditioning is unchanged.

In [ ]:
class AttnBlock(nn.Module):
    """Spatial self-attention: treat each pixel of the feature map as a token so it can see the whole image. With residual."""
    def __init__(self, ch, heads=4):
        super().__init__()
        self.norm = nn.GroupNorm(8, ch)
        self.mha = nn.MultiheadAttention(ch, heads, batch_first=True)

    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).reshape(B, C, H * W).transpose(1, 2)   # (B, HW, C)
        h, _ = self.mha(h, h, h)
        h = h.transpose(1, 2).reshape(B, C, H, W)
        return x + h


class Stage(nn.Module):
    """One resolution level: n_blocks conditional ResBlocks, with optional self-attention at the end."""
    def __init__(self, in_ch, out_ch, c_dim, n_blocks=2, attn=False):
        super().__init__()
        self.blocks = nn.ModuleList(
            [ResBlock(in_ch if i == 0 else out_ch, out_ch, c_dim) for i in range(n_blocks)])
        self.attn = AttnBlock(out_ch) if attn else None

    def forward(self, x, c):
        for b in self.blocks:
            x = b(x, c)
        if self.attn is not None:
            x = self.attn(x)
        return x


class UNet(nn.Module):
    def __init__(self, in_ch=3, base=128, c_dim=256, num_classes=None, n_blocks=2):
        super().__init__()
        self.c_dim = c_dim
        self.num_classes = num_classes
        self.time_mlp = nn.Sequential(
            nn.Linear(c_dim, c_dim), nn.SiLU(), nn.Linear(c_dim, c_dim),
        )
        if num_classes is not None:               # name embedding (+1 for the null category)
            self.label_emb = nn.Embedding(num_classes + 1, c_dim)

        self.in_conv = nn.Conv2d(in_ch, base, 3, padding=1)
        # ---- downsampling: n_blocks ResBlocks per level, attention at low-resolution layers ----
        self.down1 = Stage(base,      base,     c_dim, n_blocks)              # 96
        self.down2 = Stage(base,      base * 2, c_dim, n_blocks)              # 48
        self.down3 = Stage(base * 2,  base * 4, c_dim, n_blocks, attn=True)   # 24 +attn
        self.downsample = nn.AvgPool2d(2)
        # ---- bottleneck ----
        self.mid = Stage(base * 4, base * 4, c_dim, n_blocks, attn=True)      # 12 +attn
        # ---- upsampling ----
        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.up3 = Stage(base * 4 + base * 4, base * 2, c_dim, n_blocks, attn=True)  # 24 +attn
        self.up2 = Stage(base * 2 + base * 2, base,     c_dim, n_blocks)             # 48
        self.up1 = Stage(base     + base,     base,     c_dim, n_blocks)             # 96
        self.out = nn.Sequential(
            nn.GroupNorm(8, base), nn.SiLU(),
            nn.Conv2d(base, in_ch, 3, padding=1),
        )

    def cond(self, t, y=None, y_emb=None):
        """Compose the condition vector c = time embedding + name embedding.
        y: (B,) name index (None->null); y_emb: (B,c_dim) directly-given embedding (for fusion, takes priority)."""
        c = self.time_mlp(sinusoidal_embedding(t, self.c_dim))
        if self.num_classes is not None:
            if y_emb is None:
                if y is None:
                    y = torch.full((t.size(0),), self.num_classes,
                                   device=t.device, dtype=torch.long)
                y_emb = self.label_emb(y)
            c = c + y_emb
        return c

    def forward(self, x, t, y=None, y_emb=None):
        c = self.cond(t, y, y_emb)
        x = self.in_conv(x)
        s1 = self.down1(x, c);  x = self.downsample(s1)
        s2 = self.down2(x, c);  x = self.downsample(s2)
        s3 = self.down3(x, c);  x = self.downsample(s3)
        x = self.mid(x, c)
        x = self.upsample(x); x = self.up3(torch.cat([x, s3], 1), c)
        x = self.upsample(x); x = self.up2(torch.cat([x, s2], 1), c)
        x = self.upsample(x); x = self.up1(torch.cat([x, s1], 1), c)
        return self.out(x)

**Shape self-test** (assuming 900 classes).

In [ ]:
net = UNet(num_classes=900).to(device)
_x = torch.randn(2, 3, 96, 96, device=device)
_t = torch.randint(0, 1000, (2,), device=device)
_y = torch.randint(0, 900, (2,), device=device)
print("output shape:", tuple(net(_x, _t, _y).shape),
      "| params: %.2fM" % (sum(p.numel() for p in net.parameters()) / 1e6))

## 5. Diffusion process + loss (hand-written)

**Forward noising** (one shot, parameter-free):

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon,\qquad \varepsilon\sim\mathcal N(0,I)$$

**Training objective**: have the network predict the added noise $\varepsilon$ (the conditional version additionally passes a name $y$):

$$\mathcal L = \big\|\,\varepsilon - \varepsilon_\theta(x_t, t, y)\,\big\|^2$$

During training we also do **CFG condition dropout**: with probability `p_uncond`, replace $y$ with null.

### 🤔 Wait, shouldn't a diffusion model's loss be maximum likelihood? Why is it MSE?

Good question. **The objective at its root really is maximum likelihood**; this MSE is simplified all the way down from maximum likelihood:

**Step 1: maximum likelihood.** We want to maximize $\log p_\theta(x_0)$, but it requires integrating over the intermediate steps, which is intractable.

**Step 2: variational lower bound (ELBO).** As in a VAE, we get an optimizable upper bound:
$$-\log p_\theta(x_0)\le\mathbb{E}_q\!\left[-\log\frac{p_\theta(x_{0:T})}{q(x_{1:T}\mid x_0)}\right]=L_{\text{vlb}}$$

**Step 3: split into KL divergences.**
$$L_{\text{vlb}}=\underbrace{D_{KL}(q(x_T|x_0)\|p(x_T))}_{L_T}+\sum_{t>1}\underbrace{D_{KL}(q(x_{t-1}|x_t,x_0)\|p_\theta(x_{t-1}|x_t))}_{L_{t-1}}+\underbrace{(-\log p_\theta(x_0|x_1))}_{L_0}$$
This means "make two probability distributions agree" — make the reverse $p_\theta$ approach the true posterior $q$.

**Step 4: the Gaussian KL has a closed form** → it reduces to the squared difference of means:
$$L_{t-1}=\tfrac{1}{2\sigma_t^2}\|\tilde\mu_t-\mu_\theta\|^2+\text{const}$$

**Step 5: reparameterize using predicted noise** → the difference of means becomes a difference of noise:
$$L_{t-1}=\underbrace{\tfrac{\beta_t^2}{2\sigma_t^2\alpha_t(1-\bar\alpha_t)}}_{w_t}\|\varepsilon-\varepsilon_\theta(x_t,t)\|^2$$

**Step 6: the DDPM paper sets the weight $w_t$ to 1** → giving
$$L_{\text{simple}}=\mathbb{E}_{t,x_0,\varepsilon}\|\varepsilon-\varepsilon_\theta(x_t,t)\|^2$$
which is the MSE in the code below. The paper empirically found that dropping the weight trains more stably and gives better quality.

**Conclusion:** this MSE = the maximum-likelihood ELBO after simplification via "Gaussian KL + predicted-noise reparameterization + dropping the weight".

In [ ]:
class Diffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.T = timesteps
        self.device = device
        beta = torch.linspace(beta_start, beta_end, timesteps, device=device)
        alpha = 1.0 - beta
        self.beta = beta
        self.alpha = alpha
        self.alpha_bar = torch.cumprod(alpha, dim=0)      # ā_t

    def q_sample(self, x0, t, noise):                     # forward noising
        ab = self.alpha_bar[t][:, None, None, None]
        return ab.sqrt() * x0 + (1.0 - ab).sqrt() * noise

    def loss(self, model, x0, y, p_uncond=0.1):          # training loss (with CFG dropout)
        B = x0.size(0)
        t = torch.randint(0, self.T, (B,), device=x0.device)
        noise = torch.randn_like(x0)
        x_t = self.q_sample(x0, t, noise)
        if model.num_classes is not None and p_uncond > 0:
            y = y.clone()
            y[torch.rand(B, device=x0.device) < p_uncond] = model.num_classes  # -> null
        pred = model(x_t, t, y)
        return F.mse_loss(pred, noise)

    @torch.no_grad()
    def sample(self, model, n, y=None, y_emb=None, guidance=3.0, img_size=96, channels=3):
        model.eval()
        x = torch.randn(n, channels, img_size, img_size, device=self.device)
        use_cfg = (model.num_classes is not None) and (y is not None or y_emb is not None)
        if use_cfg:
            null = torch.full((n,), model.num_classes, device=self.device, dtype=torch.long)
        for i in reversed(range(self.T)):
            t = torch.full((n,), i, device=self.device, dtype=torch.long)
            if use_cfg:
                eps_c = model(x, t, y=y, y_emb=y_emb)
                eps_u = model(x, t, y=null)
                pred = eps_u + guidance * (eps_c - eps_u)     # CFG amplifies the condition
            else:
                pred = model(x, t)
            beta_t, alpha_t, ab_t = self.beta[i], self.alpha[i], self.alpha_bar[i]
            mean = (1.0 / alpha_t.sqrt()) * (x - (beta_t / (1.0 - ab_t).sqrt()) * pred)
            x = mean + (beta_t.sqrt() * torch.randn_like(x) if i > 0 else 0.0)
        model.train()
        return x

## 6. Data loading + experiment archiving (standard torch, no need to read closely)

The data has already been **augmented offline** (horizontal flip + in-bounds random translation); filenames look like `name__index.png`,
we use the prefix before `__` as the **name category** and return `(image, name index)`.

> Note: augmentation is already done offline, so here we **no longer add** flips/translations; the dataloader only does resize + normalization.

**Experiment archiving**: the output of each training run goes into its own folder, named
`exp<num>_<dataset>_<modules used>`, containing `checkpoints/ samples/ tb/ classes.json meta.txt`,
making it easy to compare runs.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

# ===== Experiment config: archive outputs per "experiment" =====
# folder name = exp<num>_<dataset>_<modules used>
EXP_NUM     = 5
DATASET_TAG = "pokemon20aug"            # 20 mons: front/back + normal/shiny, flip+translate aug (8538 imgs)
MODULES_TAG = "bigUNet-CFG-attn-EMA"    # larger UNet + CFG + self-attention + EMA
EXP_NAME    = f"exp{EXP_NUM:02d}_{DATASET_TAG}_{MODULES_TAG}"

DATA_DIR   = "/root/autodl-tmp/pokemon20_aug"        # ← large dataset (8538 imgs, augmentation done offline)
EXP_BASE   = "/root/autodl-tmp/experiments"
OUT_DIR    = os.path.join(EXP_BASE, EXP_NAME)
CKPT_DIR   = os.path.join(OUT_DIR, "checkpoints")
SAMPLE_DIR = os.path.join(OUT_DIR, "samples")
TB_DIR     = os.path.join(OUT_DIR, "tb")
for d in (CKPT_DIR, SAMPLE_DIR, TB_DIR):
    os.makedirs(d, exist_ok=True)

# let gpuhub's built-in TensorBoard (watching /root/tf-logs) see this experiment's logs directly
try:
    os.makedirs("/root/tf-logs", exist_ok=True)
    link = os.path.join("/root/tf-logs", EXP_NAME)
    if not os.path.exists(link):
        os.symlink(TB_DIR, link)
except OSError:
    pass

# ===== Model / training capacity config =====
BASE       = 128    # base channels (high-res layer doubled 64->128, focused on details)
N_BLOCKS   = 2      # number of ResBlocks per resolution level
EMA_DECAY  = 0.999  # weight moving-average decay (with warmup)
IMG_SIZE   = 96
BATCH_SIZE = 64

class ImageFolderFlat(Dataset):
    def __init__(self, root, img_size=96):
        self.paths = []
        for e in ("*.png", "*.jpg", "*.jpeg", "*.webp"):
            self.paths += glob.glob(os.path.join(root, "**", e), recursive=True)
        assert self.paths, f"no images found under {root}"
        names = sorted({os.path.basename(p).split("__")[0] for p in self.paths})
        self.classes = names
        self.cls2idx = {n: i for i, n in enumerate(names)}
        self.tf = transforms.Compose([                 # augmentation done offline, not added here
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),    # -> [-1,1]
        ])
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        path = self.paths[i]
        img = Image.open(path)
        if img.mode in ("RGBA", "LA", "P"):
            img = img.convert("RGBA")
            bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
            img = Image.alpha_composite(bg, img).convert("RGB")
        else:
            img = img.convert("RGB")
        label = self.cls2idx[os.path.basename(path).split("__")[0]]
        return self.tf(img), label

ds = ImageFolderFlat(DATA_DIR, IMG_SIZE)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                num_workers=4, pin_memory=True, drop_last=True)
NUM_CLASSES = len(ds.classes)

with open(os.path.join(OUT_DIR, "classes.json"), "w", encoding="utf-8") as f:
    json.dump(ds.classes, f, ensure_ascii=False, indent=1)
with open(os.path.join(OUT_DIR, "meta.txt"), "w", encoding="utf-8") as f:
    f.write(f"experiment: {EXP_NAME}\n"
            f"dataset: {DATASET_TAG}  ({len(ds)} imgs, {NUM_CLASSES} classes, front/back + shiny + aug)\n"
            f"data path: {DATA_DIR}\n"
            f"modules: {MODULES_TAG}\n"
            f"model: base={BASE}, n_blocks={N_BLOCKS}, ema_decay={EMA_DECAY}\n")
print("experiment:", EXP_NAME)
print("output dir:", OUT_DIR)
print("dataset size:", len(ds), "| number of name classes:", NUM_CLASSES)

## 7. Training (standard torch + EMA + TensorBoard, no need to read closely)

AdamW + mixed precision. New: **EMA (weight moving average)**: maintain a smoothed copy of the weights every step,
**use the EMA weights for both sampling and saving checkpoints** (on small data the sample images are steadier and sharper).
Every few epochs, sample one image for each of the 20 names, save the images + weights, and write loss and sample images to TensorBoard.

**Viewing TensorBoard**: the logs are already symlinked to `/root/tf-logs/<EXP_NAME>`; just click the
"TensorBoard" URL in the gpuhub console and select the `exp04` run.

> ⚠️ After enlarging the model, training is slower and uses more VRAM (32GB is enough). With only 587 images and no augmentation, the **overfitting risk is high** —
> watch the sample images and pick the epoch where "the samples look best and it hasn't started memorizing the training images yet"; no need to grind all the way to 300.

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import matplotlib.pyplot as plt

EPOCHS = 300; LR = 2e-4; TIMESTEPS = 1000; SAMPLE_EVERY = 10


class EMA:
    """Exponential weight moving average (with warmup): maintains a smoothed copy of the weights, used for sampling/saving.
    warmup is crucial: with small data there are few steps per epoch, and a fixed 0.999 would leave the EMA stuck at the random init for a long time,
    making samples pure noise. Use d=min(decay,(1+step)/(10+step)) so it quickly catches up to the real weights early on."""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1 + step) / (10 + step))    # warmup
        for k, v in model.state_dict().items():
            s = self.shadow[k]
            if v.dtype.is_floating_point:
                s.mul_(d).add_(v.detach(), alpha=1 - d)
            else:
                s.copy_(v)

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


writer = SummaryWriter(TB_DIR)
model     = UNet(in_ch=3, base=BASE, num_classes=NUM_CLASSES, n_blocks=N_BLOCKS).to(device)
ema       = EMA(model, decay=EMA_DECAY)
ema_model = UNet(in_ch=3, base=BASE, num_classes=NUM_CLASSES, n_blocks=N_BLOCKS).to(device)  # for sampling only
diff   = Diffusion(timesteps=TIMESTEPS, device=device)
opt    = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler("cuda")
writer.add_text("config", f"{EXP_NAME} | base={BASE} n_blocks={N_BLOCKS} ema={EMA_DECAY} "
                          f"epochs={EPOCHS} lr={LR} bs={BATCH_SIZE} T={TIMESTEPS} "
                          f"classes={NUM_CLASSES} imgs={len(ds)}")
print("params: %.2fM" % (sum(p.numel() for p in model.parameters()) / 1e6))

# evaluation generates only 1 Pikachu (model is large, generating more is slow; this one image is enough to judge progress during training)
EVAL_NAME = "pikachu"
EVAL_N    = 1
eval_idx  = ds.classes.index(EVAL_NAME) if EVAL_NAME in ds.classes else 0
eval_labels = torch.full((EVAL_N,), eval_idx, device=device)

def labeled_grid(imgs, names, ncol=4):
    """Draw a table: each cell has the Pokémon name on top and the generated image below."""
    n = len(imgs); ncol = min(ncol, n); nrow = (n + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 1.8, nrow * 2.0), squeeze=False)
    for i, ax in enumerate(axes.flatten()):
        ax.axis("off")
        if i < n:
            ax.imshow(imgs[i].permute(1, 2, 0).numpy())
            ax.set_title(names[i], fontsize=9)
    fig.tight_layout()
    return fig

step = 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    pbar = tqdm(dl, desc=f"epoch {epoch}/{EPOCHS}")
    for x0, y in pbar:
        x0 = x0.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)
        opt.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = diff.loss(model, x0, y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        ema.update(model, step)                              # update EMA (with warmup)
        writer.add_scalar("train/loss", loss.item(), step)
        step += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    if epoch % SAMPLE_EVERY == 0 or epoch == EPOCHS:
        ema.copy_to(ema_model)                              # sample using EMA weights
        imgs = (diff.sample(ema_model, EVAL_N, y=eval_labels, guidance=3.0,
                            img_size=IMG_SIZE).clamp(-1, 1) + 1) / 2
        fig = labeled_grid(imgs.cpu(), [ds.classes[eval_idx]] * EVAL_N)
        fig.savefig(os.path.join(SAMPLE_DIR, f"sample_ep{epoch}.png"),
                    dpi=120, bbox_inches="tight")
        writer.add_figure("samples", fig, epoch)
        plt.close(fig)
        torch.save(ema.shadow, os.path.join(CKPT_DIR, f"ckpt_ep{epoch}.pt"))  # save EMA weights
        print(f"  ✔ ep{epoch}: {EVAL_NAME} sample + checkpoint saved to {EXP_NAME}")

writer.close()

## ✅ Next steps

- All artifacts of this run are under `experiments/<EXP_NAME>/`:
  `checkpoints/` (weights), `samples/` (sample images), `tb/` (TensorBoard logs), `classes.json`, `meta.txt`.
- **View training curves / samples**: `tensorboard --logdir /root/autodl-tmp/experiments --host 0.0.0.0 --port 6007`
- **Generate / fuse** → open `pokemon_gen.ipynb` and change its `EXP_NAME` to this run's name.
- **For your next experiment**: just change `EXP_NUM` / `DATASET_TAG` / `MODULES_TAG` in the top data cell of this notebook,
  and the output will automatically go to a new experiment folder without overwriting anything.